# FordRetain — Clustering de Perfis (Base 1)

Identifica os **4 perfis comportamentais** usando K-Means.

**Regra crítica:** Esta etapa usa a Base 1 com dados históricos completos (incluindo pós-compra).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import joblib
import sys
import os
sys.path.append('../src')
from preprocessamento import carregar_dados, construir_base1

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../models', exist_ok=True)

df = carregar_dados('../dados/vin_share_Desafio_02.xlsx')
base1 = construir_base1(df)
print(f'Base 1 pronta: {base1.shape}')
base1.head()

In [ ]:
features_clustering = [
    'total_revisoes',
    'intervalo_medio_dias',
    'dias_desde_ultima_revisao',
    'km_atual',
    'idade_meses',
    'garantia_expirada',
    'abandonou_apos_1a'
]

X = base1[features_clustering].dropna()
print(f'Amostras para clustering: {X.shape[0]:,}')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
inertias = []
silhouettes = []
Ks = range(2, 8)

for k in Ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(Ks), inertias, 'bo-')
axes[0].set_title('Método do Cotovelo')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inércia')
axes[1].plot(list(Ks), silhouettes, 'ro-')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('K')
plt.tight_layout()
plt.savefig('../outputs/05_escolha_k.png', dpi=150)
plt.show()

In [ ]:
K_FINAL = 4
kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

base1_clust = X.copy()
base1_clust['cluster'] = labels

perfil_clusters = base1_clust.groupby('cluster')[features_clustering].mean().round(2)
print('Perfil médio de cada cluster:')
print(perfil_clusters)

In [ ]:
# Analise perfil_clusters acima e ajuste este mapeamento se necessário.
# Lógica:
#   MUITAS revisões + intervalos CURTOS  → Fiel
#   Revisões MÉDIAS + sensível a preço   → Econômico
#   Intervalos LONGOS + poucas revisões  → Esquecido
#   APENAS 1 revisão + garantia expirada → Abandono

MAPA_PERFIS = {0: 'Fiel', 1: 'Econômico', 2: 'Esquecido', 3: 'Abandono'}

base1_clust['perfil'] = base1_clust['cluster'].map(MAPA_PERFIS)
print('Distribuição dos perfis:')
print(base1_clust['perfil'].value_counts())

joblib.dump(kmeans, '../models/kmeans_perfis.pkl')
joblib.dump(scaler, '../models/scaler_base1.pkl')
base1_clust.to_csv('../models/base1_com_perfis.csv', index=False)
print('Modelos salvos em ../models/')

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

cores = {'Fiel': '#27AE60', 'Econômico': '#E67E22', 'Esquecido': '#3498DB', 'Abandono': '#E74C3C'}

plt.figure(figsize=(10, 7))
for perfil, cor in cores.items():
    mask = (base1_clust['perfil'] == perfil).values
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=cor, label=perfil, alpha=0.4, s=10)
plt.title('PCA — 4 Perfis de Cliente FordRetain')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/06_clusters_pca.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (perfil, cor) in enumerate(cores.items()):
    dados = base1_clust[base1_clust['perfil'] == perfil]
    axes[i].bar(
        features_clustering,
        dados[features_clustering].mean(),
        color=cor, alpha=0.8
    )
    axes[i].set_title(f'Perfil: {perfil} (n={len(dados):,})')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Características Médias por Perfil', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/07_perfis_radar.png', dpi=150)
plt.show()